# MalXplain
### An Improved Lightweight and Interpretable Machine Learning Approach for Detecting New Obfuscated Malware Variants

**Software Engineering, Phase 2 implementation notebook**

Pravara Jain (K020) · Yash Mehta (K035) · Somya Soni (K063)  
B. Tech. Computer Engineering, Cyber Security, MPSTME, NMIMS Mumbai

---

This notebook runs the complete MalXplain pipeline. Each stage has its own cell, so any
single stage can be run and inspected on its own.

The project reproduces Madamidola, Ngobigha and Ez-zizi (2024), which trains a malware
detector on **one** malware subtype and then tests it against the **fourteen subtypes it has
never seen**, and then adds four enhancements that address limitations the authors state
explicitly in that paper.

Repository: https://github.com/spy-ka-y/malxplain  
Live dashboard: https://malxplain.streamlit.app

**Two things to know before running.**

1. `FAST_MODE` in the setup cell controls how the notebook runs. Left at `True` it loads the
   results committed to the repository, so every figure and table below is reproduced in
   about a minute. Set it to `False` to retrain everything from the raw dataset, which takes
   roughly thirty minutes on a Colab runtime.
2. The reporting layer uses a language model hosted on the same machine. Colab has no such
   service, so that stage falls back to a deterministic template and says so in its output.
   The language model path is demonstrated locally.

## 0. Setup

Clone the repository and install the dependencies. The dataset is committed to the
repository, so there is no separate download step.

In [ ]:
import os, sys, subprocess

REPO = 'https://github.com/spy-ka-y/malxplain.git'
ROOT = '/content/malxplain'

if not os.path.exists(ROOT):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, ROOT], check=True)
else:
    print('Repository already present.')

os.chdir(ROOT)
sys.path.insert(0, ROOT)
print('Working directory:', os.getcwd())

In [ ]:
!pip install -q shap xgboost
print('Dependencies installed.')

In [ ]:
# Set to False to retrain every stage from the raw dataset (about 30 minutes).
FAST_MODE = True

import pandas as pd, numpy as np, json
import matplotlib.pyplot as plt
import sklearn, shap

print('pandas      ', pd.__version__)
print('numpy       ', np.__version__)
print('scikit-learn', sklearn.__version__)
print('shap        ', shap.__version__)
print()
print('FAST_MODE =', FAST_MODE)

## 1. Data preparation

`src/preprocessing.py` loads CIC-MalMem-2022 and refuses to continue unless the record
count matches the figure published in the paper. It then removes the features that hold an
identical value for every record, which carry no information, and scales what remains.

The paper reports 58,596 records and 52 usable features after this step. Both are checked
here rather than assumed.

In [ ]:
from src.preprocessing import prepare, load_raw, find_invariant_features

raw = load_raw()
invariant = find_invariant_features(raw)

print('Raw records          :', f'{len(raw):,}')
print('Raw numeric features :', raw.select_dtypes(include="number").shape[1])
print('Invariant features   :', invariant)
print()

ds = prepare(verbose=False)
print('Prepared records     :', f'{len(ds):,}')
print('Usable features      :', len(ds.feature_names))
print('Benign / malware     :', int((ds.y == 0).sum()), '/', int((ds.y == 1).sum()))
print('Malware subtypes     :', len(ds.subtypes))
print()
print(ds.subtypes)

## 2. Stage 1, the baseline comparison

The first part of the base paper compares seven classifiers on a conventional random split,
in which every malware family appears in both halves, and identifies Random Forest as the
strongest. This stage reproduces that comparison and establishes the numerical baseline
against which the reproduction is judged.

In [ ]:
if FAST_MODE:
    stage1 = pd.read_csv('results/stage1_baseline_holdout.csv', index_col=0)
else:
    from src.baseline import run_holdout
    stage1 = run_holdout(ds)

stage1.sort_values('Accuracy', ascending=False)

## 3. Stage 2, the withheld subtype detectors

This is the contribution of the base paper and the core of the project. For each of the
fifteen malware subtypes the pipeline builds a training set containing **only that subtype**
plus an equal number of benign records, selects the five most important features inside that
training split alone, trains a Random Forest, and then evaluates it against the fourteen
subtypes it never saw.

The paper reports that eleven of the fifteen detectors exceed 99 percent accuracy on the
unseen data.

In [ ]:
if FAST_MODE:
    stage2 = pd.read_csv('results/stage2_adaptive.csv')
else:
    from src.adaptive import run_all
    stage2 = run_all(ds)

above_99 = (stage2['Accuracy'] >= 0.99).sum()
print(f'Detectors above 99 percent accuracy on unseen subtypes: {above_99} of {len(stage2)}')
print(f'Weakest subtype: {stage2.loc[stage2["Accuracy"].idxmin(), "Subtype"]}')
print()
stage2.sort_values('Accuracy', ascending=False).head(20)

In [ ]:
from src.charts import reproduction_chart
fig = reproduction_chart(stage2)
fig

## 4. Enhancement 1, hyperparameter tuning

The base paper used library defaults throughout and lists this as its first limitation.

The search is cross validated **inside the training split only**. The unseen subtype holdout
is never used for model selection, which is what keeps the reported gain honest rather than
an artefact of tuning against the test set.

In [ ]:
if FAST_MODE:
    tuning = pd.read_csv('results/upgrade1_tuning.csv')
else:
    from src.tuning import main as run_tuning
    run_tuning()
    tuning = pd.read_csv('results/upgrade1_tuning.csv')

improved = (tuning['Δ unseen recall'] > 0).sum()
print(f'Detectors improved on unseen recall: {improved} of {len(tuning)}')
print(f'Mean change: {tuning["Δ unseen recall"].mean() * 100:+.3f} percentage points')
print()
tuning.sort_values('Δ unseen recall', ascending=False).head(10)

In [ ]:
from src.charts import tuning_chart
fig = tuning_chart(tuning)
fig

## 5. Enhancement 2, comparing classifier families

The base paper tests seven classifiers on known malware but evaluates only Random Forest on
unseen subtypes, so it cannot say whether the generalisation it observes belongs to Random
Forest specifically or to the protocol in general.

This stage runs the identical withheld subtype protocol across five classifier families and
answers that question.

In [ ]:
if FAST_MODE:
    comparison = pd.read_csv('results/upgrade2_comparison_summary.csv', index_col=0)
else:
    from src.comparison import main as run_comparison
    run_comparison()
    comparison = pd.read_csv('results/upgrade2_comparison_summary.csv', index_col=0)

comparison.sort_values('Mean_unseen_recall', ascending=False)

In [ ]:
from src.charts import classifier_chart
fig = classifier_chart(comparison)
fig

## 6. Enhancement 3, cross subtype error analysis

The base paper reports aggregate accuracy and lists the absence of error analysis as a
limitation. This stage builds the complete fifteen by fifteen matrix of every detector
against every subtype.

This is where the project's own finding comes from. An aggregate figure above 99 percent
hides the fact that one subtype is detected far less reliably than the rest.

In [ ]:
recall = pd.read_csv('results/upgrade3_cross_subtype_recall.csv', index_col=0)
hardest = pd.read_csv('results/upgrade3_hardest_subtypes.csv', index_col=0)

print('Hardest subtypes to detect when unseen:')
print()
hardest.head(6)

In [ ]:
from src.charts import blindspot_heatmap
fig = blindspot_heatmap(recall)
fig

In [ ]:
worst = pd.read_csv('results/upgrade3_worst_blindspots.csv')
print('The individual detector and subtype pairs that fail worst:')
print()
worst.head(8)

In [ ]:
diagnosis = pd.read_csv('results/upgrade3_blindspot_diagnosis.csv')
print('Feature level diagnosis, missed samples against caught samples:')
print()
diagnosis

## 7. Enhancement 4, SHAP explainability

The base paper's explainability output is a plot that a human has to interpret. This stage
computes the SHAP attribution for a single prediction and serialises it into a structured
payload, which turns an explanation from a picture into data that another component can
consume.

This cell trains the reference detector, so it takes about half a minute even in fast mode.

In [ ]:
from src.explainability import build_explainer, explain_instance

ctx = build_explainer(ds, subtype='Transponder')
print('Reference detector trained on :', ctx['subtype'])
print('Features used                 :', ctx['features'])
print('Holdout samples available     :', f'{len(ctx["X_test"]):,}')

In [ ]:
# Pick a sample whose true subtype is NOT the one the detector was trained on,
# which is the zero day analogue.
unseen = ctx['test_subtype'][ctx['test_subtype'] != ctx['subtype']]
unseen = unseen[unseen != 'Benign']
index = unseen.index[0]

explanation = explain_instance(ctx, index)

print('Detector trained on :', explanation['detector_trained_on'])
print('True subtype        :', explanation['actual_subtype'])
print('Seen in training    :', explanation['seen_in_training'])
print('Verdict             :', explanation['prediction'])
print('Confidence          :', f"{explanation['confidence']:.4f}")
print('Ground truth        :', explanation['ground_truth'])
print()
pd.DataFrame(explanation['evidence'])

In [ ]:
from src.charts import shap_bar
fig = shap_bar(pd.DataFrame(explanation['evidence']))
fig

## 8. The reporting layer

The structured payload above is handed to a language model hosted on the same machine, which
turns it into a readable analyst finding. The payload deliberately excludes the ground truth
label, so the model cannot restate an answer it was given.

**On Colab there is no local model service**, so the call below falls back to a deterministic
template and reports `source = 'template'`. Running the same cell on a machine with Ollama
returns `source = 'llm'` instead. Both paths are shown so the fallback can be verified rather
than taken on trust.

In [ ]:
from src.reporter import ollama_available, generate_finding, build_prompt

print('Local model service reachable:', ollama_available())
print()

result = generate_finding(explanation)
print('Source :', result['source'])
if result.get('error'):
    print('Reason :', result['error'])
print()
print(result['finding'])

In [ ]:
# The exact evidence the model receives. Nothing else is sent.
print(build_prompt(explanation))

## 9. Verification

Every claim the project makes is checked by a module that recomputes the figure rather than
restating it. The guards assert that the training and test sets are disjoint across all
fifteen splits and that every training set is balanced equally between benign and malicious
records, so that any improvement claimed is a real effect and not an artefact of leakage.

In [ ]:
if FAST_MODE:
    print(open('results/verification_report.txt').read())
else:
    from src.verify import build_report
    build_report()
    print(open('results/verification_report.txt').read())

## 10. Summary

What this notebook has shown, in order:

1. The dataset matches the published description exactly, 58,596 records reducing to 52 usable features.
2. Stage 1 reproduces the base paper's classifier comparison.
3. Stage 2 reproduces its central result, eleven of fifteen detectors above 99 percent on subtypes they never saw.
4. Hyperparameter tuning, which the paper omitted, improves unseen subtype recall.
5. Running the protocol across five classifier families answers a question the paper could not.
6. The cross subtype matrix exposes a blind spot that aggregate accuracy hides.
7. SHAP attribution is serialised into a structured payload rather than left as a plot.
8. That payload becomes a written finding, with a verifiable fallback when no model service is present.
9. The verification module confirms there is no leakage behind any of it.

---

Repository: https://github.com/spy-ka-y/malxplain  
Live dashboard: https://malxplain.streamlit.app